In [1]:
import os
import tensorflow as tf
import pandas as pd
import sys
sys.path.append(os.path.join('c:\\', *os.getcwd().split('\\')[1:-1]))
from sgcc7 import *

In [2]:

# set the parameter bounds
param_bounds = {
    "fts": [0, 175],
    "t": [40, 125],
    "ampc": [0.01, 0.1],
    'ampm': [0,3],
    "ampg": [0.1, 4],
    "ampw": [0.01, 0.08],
    "d": [10, 40],
    "inh_d": [0, 40],
    "inh_w": [0, 3],
}

In [17]:
from itertools import permutations
perm_choices = np.array(list(permutations(permutations(np.arange(3)), 2)))
P = tf.one_hot(
    perm_choices,
    3,
    axis=-1,
)
P = tf.transpose(P, (0,1,3,2))

In [33]:
import matplotlib.pyplot as plt
# P has shape (30, 2, 3, 3)
# You want weights of shape (n_samples, 7, 30) 
# so that after weighting and summing over the 30 permutations
# you get (n_samples, 7, 2, 3, 3)
n_samples = 100
# Learnable logits drawn from normal distribution
# shape: (n_samples, 7, 30)
logits = tf.Variable(
    tf.random.normal(shape=(n_samples, 7, 30), mean=0.0, stddev=1.0)
)

# Convert to probabilities via softmax over the 30 permutations
probs = tf.nn.softmax(logits, axis=-1)  # shape: (n_samples, 7, 30)

# Reshape P for broadcasting
# P: (30, 2, 3, 3) -> (1, 1, 30, 2, 3, 3)
P_expanded = tf.reshape(P, (1, 1, 30, 2, 3, 3))

# Reshape probs for broadcasting
# probs: (n_samples, 7, 30) -> (n_samples, 7, 30, 1, 1, 1)
probs_expanded = tf.reshape(probs, (n_samples, 7, 30, 1, 1, 1))

# Weighted sum over the 30 permutations
# Result shape: (n_samples, 7, 2, 3, 3)
soft_P = tf.reduce_sum(probs_expanded * P_expanded, axis=2)

In [34]:
soft_P

<tf.Tensor: shape=(100, 7, 2, 3, 3), dtype=float32, numpy=
array([[[[[0.33568344, 0.42851847, 0.23579817],
          [0.2984167 , 0.30990762, 0.3916757 ],
          [0.36589992, 0.261574  , 0.37252617]],

         [[0.269156  , 0.30959013, 0.42125395],
          [0.46045938, 0.21227911, 0.32726154],
          [0.2703847 , 0.4781308 , 0.25148457]]],


        [[[0.51757383, 0.159656  , 0.32277015],
          [0.13184018, 0.52600586, 0.3421539 ],
          [0.35058594, 0.31433806, 0.3350759 ]],

         [[0.29573095, 0.36646515, 0.33780378],
          [0.42661476, 0.33207065, 0.24131453],
          [0.2776542 , 0.30146414, 0.42088163]]],


        [[[0.18122143, 0.39951104, 0.41926754],
          [0.43325073, 0.28217342, 0.28457582],
          [0.38552785, 0.31831557, 0.29615664]],

         [[0.281872  , 0.379118  , 0.33900997],
          [0.33926013, 0.23503123, 0.42570862],
          [0.37886783, 0.38585073, 0.23528138]]],


        ...,


        [[[0.3134332 , 0.18320803, 0.5033588

In [39]:
tf.argmax(soft_P, axis=2)

<tf.Tensor: shape=(100, 7, 3, 3), dtype=int64, numpy=
array([[[[0, 0, 1],
         [1, 0, 0],
         [0, 1, 0]],

        [[0, 1, 1],
         [1, 0, 0],
         [0, 0, 1]],

        [[1, 0, 0],
         [0, 0, 1],
         [0, 1, 0]],

        ...,

        [[0, 1, 0],
         [1, 0, 1],
         [0, 1, 1]],

        [[0, 1, 0],
         [1, 0, 1],
         [1, 0, 0]],

        [[0, 1, 0],
         [1, 0, 1],
         [1, 0, 1]]],


       [[[0, 1, 0],
         [0, 0, 1],
         [1, 1, 0]],

        [[0, 1, 0],
         [1, 0, 1],
         [0, 1, 0]],

        [[1, 0, 1],
         [1, 0, 1],
         [0, 1, 0]],

        ...,

        [[0, 0, 1],
         [1, 1, 0],
         [1, 0, 0]],

        [[1, 0, 0],
         [0, 1, 1],
         [1, 0, 0]],

        [[1, 0, 0],
         [0, 1, 0],
         [0, 0, 1]]],


       [[[1, 0, 0],
         [1, 1, 0],
         [0, 0, 1]],

        [[1, 0, 0],
         [1, 1, 0],
         [0, 1, 1]],

        [[0, 1, 0],
         [1, 0, 0],
      

In [3]:
# initialize X and Y
X = tf.convert_to_tensor([0.02,0.04,0.08,0.1,0.12,0.16,0.2,0.24,0.28,0.32], dtype = tf.float32)

v1_xs_file = os.path.join('c:\\', *os.getcwd().split('\\')[1:-1], 'project_datafiles', 'v1_ori_phase_condition_pcascores_wcomp.pkl')
v1_scores = pd.read_pickle(v1_xs_file)
v1_scores_condition_averaged = np.array([np.array(x) for x in v1_scores.scores.values]).mean(0)
Y_true = v1_scores_condition_averaged[:2,:,:].transpose(0,2,1)

In [4]:
model = SGCCircuit(bounds=param_bounds)
model.initialize_random_parameters(2,3,10)
model.update_transform()

In [8]:
model.sinkhorn(model.soft_perm, temperature=0.01)[0,0,0]

<tf.Tensor: shape=(3, 3), dtype=float32, numpy=
array([[0.0000000e+00, 5.0000001e-02, 1.0000000e+00],
       [0.0000000e+00, 9.4999993e-01, 0.0000000e+00],
       [1.0000000e+00, 4.9937743e-16, 5.9664670e-24]], dtype=float32)>

In [6]:
optimizer = Optimize(model, epochs=1000, loss_threshold=0)
_=optimizer.fit(X,Y_true)

Optimizer initialized with <keras.optimizers.optimizer_v2.adam.Adam object at 0x000001BE091702E0>
Training step = 0, N_exploration_samples = 10,
min_loss = 3.2221016883850098
med_loss = 21.153892517089844
max_loss = 50.05815505981445

Training step = 100, N_exploration_samples = 10,
min_loss = 1.925546646118164
med_loss = 16.274282455444336
max_loss = 40.72875213623047

Training step = 200, N_exploration_samples = 10,
min_loss = 1.1820331811904907
med_loss = 12.651655197143555
max_loss = 33.19948959350586

Training step = 300, N_exploration_samples = 10,
min_loss = 0.747680127620697
med_loss = 9.911554336547852
max_loss = 27.11911964416504

Training step = 400, N_exploration_samples = 10,
min_loss = 0.4983401298522949
med_loss = 7.7526750564575195
max_loss = 22.19786262512207

Training step = 500, N_exploration_samples = 10,
min_loss = 0.35438066720962524
med_loss = 6.060492038726807
max_loss = 18.20754051208496

Training step = 600, N_exploration_samples = 10,
min_loss = 0.26799398660

In [7]:
model.sinkhorn(model.soft_perm, temperature=0.001)[0,0,0]

<tf.Tensor: shape=(3, 3), dtype=float32, numpy=
array([[0.       , 0.05     , 1.       ],
       [0.       , 0.9499999, 0.       ],
       [1.       , 0.       , 0.       ]], dtype=float32)>